# Clase 12 — Práctica: cierra tu propio caso de uso NOAA/JFK

Esta es la práctica final del curso: vas a extender el pipeline de la clase con tu
propio análisis, igual que harías en un proyecto real.

## Ejercicio 1 — Explorar una variable adicional

**Enunciado:** usando el DataFrame `clima` ya limpio de la clase, dibuja la
tendencia horaria de `HOURLYRelativeHumidity` (humedad relativa) durante el año 2017,
y calcula su correlación con `HOURLYDRYBULBTEMPF` (temperatura).

**Pista:** usa `.loc["2017", columna]` para filtrar el año, y `.corr()` entre dos
columnas de un DataFrame (o `Series.corr(Series)`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/fJKLoMKtgpDAl5MkA6rf2A/jfk-weather.csv"
crudo = pd.read_csv(url, parse_dates=["DATE"], low_memory=False)
columnas = ["DATE", "HOURLYDRYBULBTEMPF", "HOURLYPrecip",
            "HOURLYWindSpeed", "HOURLYRelativeHumidity", "HOURLYStationPressure"]
clima = crudo[columnas].copy()
clima = clima.set_index(pd.DatetimeIndex(clima["DATE"])).drop(columns=["DATE"])
clima["HOURLYPrecip"] = clima["HOURLYPrecip"].replace("T", "0.00")
clima["HOURLYPrecip"] = clima["HOURLYPrecip"].astype(str).str.replace("s", "", regex=False)
for columna in clima.columns:
    clima[columna] = pd.to_numeric(
        clima[columna].astype(str).str.replace("*", "", regex=False), errors="coerce"
    )
clima = clima.ffill().dropna()

# TODO: dibuja la tendencia de HOURLYRelativeHumidity durante 2017

# TODO: calcula la correlación entre HOURLYRelativeHumidity y HOURLYDRYBULBTEMPF
correlacion_humedad_temp = None

print(f"Correlación humedad-temperatura: {correlacion_humedad_temp:.2f}")


<details>
<summary>💡 Ver solución</summary>

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/fJKLoMKtgpDAl5MkA6rf2A/jfk-weather.csv"
crudo = pd.read_csv(url, parse_dates=["DATE"], low_memory=False)
columnas = ["DATE", "HOURLYDRYBULBTEMPF", "HOURLYPrecip",
            "HOURLYWindSpeed", "HOURLYRelativeHumidity", "HOURLYStationPressure"]
clima = crudo[columnas].copy()
clima = clima.set_index(pd.DatetimeIndex(clima["DATE"])).drop(columns=["DATE"])
clima["HOURLYPrecip"] = clima["HOURLYPrecip"].replace("T", "0.00")
clima["HOURLYPrecip"] = clima["HOURLYPrecip"].astype(str).str.replace("s", "", regex=False)
for columna in clima.columns:
    clima[columna] = pd.to_numeric(
        clima[columna].astype(str).str.replace("*", "", regex=False), errors="coerce"
    )
clima = clima.ffill().dropna()

clima.loc["2017", "HOURLYRelativeHumidity"].plot(figsize=(10, 4))
plt.title("Humedad relativa horaria en JFK durante 2017")
plt.ylabel("% de humedad")
plt.show()

correlacion_humedad_temp = clima["HOURLYRelativeHumidity"].corr(clima["HOURLYDRYBULBTEMPF"])
print(f"Correlación humedad-temperatura: {correlacion_humedad_temp:.2f}")
```
</details>

## Ejercicio 2 — ¿Puedes superar al modelo de referencia?

**Enunciado:** en la clase comparaste el modelo "sin cambio" contra la media móvil de
3 horas. Ahora prueba con una media móvil de **6 horas** y compara su RMSE contra
ambos modelos anteriores. ¿Cuál de los tres funciona mejor?

**Pista:** el código es idéntico al de la media móvil de 3 horas, solo cambia el
número dentro de `.rolling(...)`.

In [ ]:
from sklearn.metrics import mean_squared_error

temperatura = clima["HOURLYDRYBULBTEMPF"]

prediccion_baseline = temperatura.shift(1).dropna()
rmse_baseline = np.sqrt(mean_squared_error(
    temperatura.loc[prediccion_baseline.index], prediccion_baseline
))

prediccion_media_3h = temperatura.rolling(3).mean().shift(1).dropna()
rmse_media_3h = np.sqrt(mean_squared_error(
    temperatura.loc[prediccion_media_3h.index], prediccion_media_3h
))

# TODO: calcula prediccion_media_6h con .rolling(6).mean().shift(1).dropna()
prediccion_media_6h = None

# TODO: calcula su RMSE de la misma forma que los dos modelos anteriores
rmse_media_6h = None

print(f"RMSE sin cambio:      {rmse_baseline:.2f} °F")
print(f"RMSE media móvil 3h:  {rmse_media_3h:.2f} °F")
print(f"RMSE media móvil 6h:  {rmse_media_6h:.2f} °F")


<details>
<summary>💡 Ver solución</summary>

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/fJKLoMKtgpDAl5MkA6rf2A/jfk-weather.csv"
crudo = pd.read_csv(url, parse_dates=["DATE"], low_memory=False)
columnas = ["DATE", "HOURLYDRYBULBTEMPF", "HOURLYPrecip",
            "HOURLYWindSpeed", "HOURLYRelativeHumidity", "HOURLYStationPressure"]
clima = crudo[columnas].copy()
clima = clima.set_index(pd.DatetimeIndex(clima["DATE"])).drop(columns=["DATE"])
clima["HOURLYPrecip"] = clima["HOURLYPrecip"].replace("T", "0.00")
clima["HOURLYPrecip"] = clima["HOURLYPrecip"].astype(str).str.replace("s", "", regex=False)
for columna in clima.columns:
    clima[columna] = pd.to_numeric(
        clima[columna].astype(str).str.replace("*", "", regex=False), errors="coerce"
    )
clima = clima.ffill().dropna()

from sklearn.metrics import mean_squared_error

temperatura = clima["HOURLYDRYBULBTEMPF"]

prediccion_baseline = temperatura.shift(1).dropna()
rmse_baseline = np.sqrt(mean_squared_error(
    temperatura.loc[prediccion_baseline.index], prediccion_baseline
))

prediccion_media_3h = temperatura.rolling(3).mean().shift(1).dropna()
rmse_media_3h = np.sqrt(mean_squared_error(
    temperatura.loc[prediccion_media_3h.index], prediccion_media_3h
))

prediccion_media_6h = temperatura.rolling(6).mean().shift(1).dropna()
rmse_media_6h = np.sqrt(mean_squared_error(
    temperatura.loc[prediccion_media_6h.index], prediccion_media_6h
))

print(f"RMSE sin cambio:      {rmse_baseline:.2f} °F")
print(f"RMSE media móvil 3h:  {rmse_media_3h:.2f} °F")
print(f"RMSE media móvil 6h:  {rmse_media_6h:.2f} °F")
```

Normalmente, ventanas más largas suavizan demasiado la predicción y empeoran el
RMSE frente a ventanas cortas — es un buen ejemplo de que "más complejo" no siempre
significa "mejor". ¡Enhorabuena por completar el curso!
</details>